# 03.5 · Naive Bayes: counting your way to a prediction

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/03-classification/05-naive-bayes/lesson.ipynb)

Read the [README](README.md) and do the ten-row table by hand first. Here you check every fraction, let scikit-learn reproduce them, then use Naive Bayes where it shines: text.

## 1. The ten customers

In [1]:
import pandas as pd, numpy as np
import warnings; warnings.filterwarnings("ignore")

ten = pd.DataFrame({
    "visited":      [1, 1, 1, 1, 0, 1, 0, 0, 0, 1],
    "opened_email": [1, 1, 0, 1, 1, 0, 0, 0, 1, 0],
    "churned":      [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
}, index=range(1, 11))
ten

,visited,opened_email,churned
1,1,1,0
2,1,1,0
3,1,0,0
4,1,1,0
5,0,1,0
6,1,0,0
7,0,0,1
8,0,0,1
9,0,1,1
10,1,0,1


## 2. Priors and likelihoods, counted

In [2]:
prior = ten["churned"].value_counts(normalize=True).sort_index()
print("P(stay) =", prior[0], "  P(churn) =", prior[1])

lik = {}
for c in [0, 1]:
    sub = ten[ten["churned"] == c]
    lik[c] = {"visited=0": (sub["visited"] == 0).mean(), "opened_email=0": (sub["opened_email"] == 0).mean()}
pd.DataFrame(lik, columns=[1, 0]).rename(columns={1: "among churners", 0: "among stayers"}).round(3)

P(stay) = 0.6   P(churn) = 0.4


,among churners,among stayers
visited=0,0.75,0.167
opened_email=0,0.75,0.333


0.75 / 0.167 and 0.75 / 0.333, as in the README.

## 3. Bayes' theorem on one feature

A new customer did not visit. $P(\text{churn} \mid V = 0)$ by the formula, and by counting directly.

In [3]:
p_novisit_given_churn = lik[1]["visited=0"]
p_novisit = (ten["visited"] == 0).mean()
bayes = p_novisit_given_churn * prior[1] / p_novisit
direct = ten[ten["visited"] == 0]["churned"].mean()
print(f"Bayes: {p_novisit_given_churn} × {prior[1]} / {p_novisit} = {bayes:.2f}")
print(f"Direct count among non-visitors: {direct:.2f}")

Bayes: 0.75 × 0.4 / 0.4 = 0.75
Direct count among non-visitors: 0.75


## 4. Naive Bayes on two features: no visit, no email

In [4]:
score_churn = prior[1] * lik[1]["visited=0"] * lik[1]["opened_email=0"]
score_stay = prior[0] * lik[0]["visited=0"] * lik[0]["opened_email=0"]
print(f"score(churn) = {prior[1]} × {lik[1]['visited=0']} × {lik[1]['opened_email=0']} = {score_churn:.4f}")
print(f"score(stay)  = {prior[0]} × {lik[0]['visited=0']:.3f} × {lik[0]['opened_email=0']:.3f} = {score_stay:.4f}")
print(f"P(churn | no visit, no email) = {score_churn / (score_churn + score_stay):.3f}")

score(churn) = 0.4 × 0.75 × 0.75 = 0.2250
score(stay)  = 0.6 × 0.167 × 0.333 = 0.0333
P(churn | no visit, no email) = 0.871


0.87, predict churn. scikit-learn's `CategoricalNB` with `alpha=0` (no smoothing) must agree exactly.

In [5]:
from sklearn.naive_bayes import CategoricalNB
X10, y10 = ten[["visited", "opened_email"]], ten["churned"]
nb0 = CategoricalNB(alpha=0).fit(X10, y10)
print("no smoothing:", nb0.predict_proba([[0, 0]]).round(3))
nb1 = CategoricalNB(alpha=1).fit(X10, y10)
print("Laplace (+1): ", nb1.predict_proba([[0, 0]]).round(3))

no smoothing: [[0.129 0.871]]
Laplace (+1):  [[0.24 0.76]]


With smoothing the answer softens a little (every fraction moves toward 1/2) but the decision is the same. Smoothing matters when a count is **zero**:

In [6]:
# Pretend no stayer ever left an email unopened: P(opened_email=0 | stay) = 0
ten_zero = ten.copy()
ten_zero.loc[ten_zero["churned"] == 0, "opened_email"] = 1
for alpha in [0, 1]:
    m = CategoricalNB(alpha=alpha).fit(ten_zero[["visited", "opened_email"]], ten_zero["churned"])
    print(f"alpha={alpha}: P(stay | visited=1, opened_email=0) = {m.predict_proba([[1, 0]])[0, 0]:.3f}")

alpha=0: P(stay | visited=1, opened_email=0) = 0.000
alpha=1: P(stay | visited=1, opened_email=0) = 0.388


Without smoothing a single unseen combination makes "stay" impossible (probability 0.000) even though the customer *visited*, which is strong evidence of staying. With +1 smoothing, the evidence is weighed sensibly.

## 5. Gaussian NB: continuous features

For numbers like `days_since_last`, Naive Bayes fits a bell curve per class (mean and standard deviation), and the likelihood is the height of that curve at the customer's value.

In [7]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

cust20 = pd.read_csv(data_path("shopwave_customers_small.csv"))
X20 = cust20[["visits_last_month", "days_since_last"]]
y20 = cust20["churned"]
cust20

,customer_id,visits_last_month,items_last_month,days_since_last,country,spend_next_month,churned
0,C001,2,2,55,UK,27.61,1
1,C002,11,8,44,FR,220.42,0
2,C003,10,6,22,DE,176.21,0
3,C004,7,6,58,UK,127.58,1
4,C005,7,5,25,FR,150.95,1
5,C006,13,7,20,FR,250.03,0
6,C007,2,2,54,UK,47.33,1
7,C008,10,5,22,UK,196.92,0
8,C009,3,3,5,DE,94.82,1
9,C010,2,1,28,UK,45.05,1


In [8]:
from sklearn.naive_bayes import GaussianNB
g = GaussianNB().fit(X20, y20)
print("class means (visits, days_since_last):"); print(pd.DataFrame(g.theta_, index=["stay", "churn"], columns=X20.columns).round(1))
print("\nclass std devs:"); print(pd.DataFrame(np.sqrt(g.var_), index=["stay", "churn"], columns=X20.columns).round(1))
new = pd.DataFrame({"visits_last_month": [3, 12], "days_since_last": [50, 10]})
print("\nP(churn) for a 3-visit/50-day customer and a 12-visit/10-day customer:", g.predict_proba(new)[:, 1].round(3))

class means (visits, days_since_last):
       visits_last_month  days_since_last
stay                10.7             26.0
churn                5.0             35.6

class std devs:
       visits_last_month  days_since_last
stay                 1.9             11.2
churn                3.4             18.0

P(churn) for a 3-visit/50-day customer and a 12-visit/10-day customer: [1.    0.043]


Churners average fewer visits and more days since their last order; a customer far from the churn means on both features gets a low churn probability.

## 6. On the real ShopWave customers

Gaussian NB against the logistic regression from 03.4, both scored with the 03.6 metrics on held-out customers.

> **About `train_test_split`.** We hold back 25% of customers and score the model only on those, because scoring a model on the rows it learned from is like marking your own homework. Module 04 explains this properly; for now, just know that `X_test` / `y_test` are "exam" customers the model never saw.

In [9]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd, numpy as np
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail
from sklearn.model_selection import train_test_split

xlsx = ensure_online_retail()
csv = xlsx.with_name("online_retail_2010_2011.csv")          # fast cache written by lesson 01.3
raw = pd.read_csv(csv, parse_dates=["InvoiceDate"]) if csv.exists() else pd.read_excel(xlsx, sheet_name="Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

3,317 customers, churn rate 43.6%, train 2,487 / test 830


In [10]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_train, y_train)
proba_test = logreg.predict_proba(X_test)[:, 1]          # P(churn) for each test customer
pred_test = logreg.predict(X_test)                        # same as (proba_test >= 0.5)

from sklearn.metrics import classification_report, f1_score
gnb = GaussianNB().fit(X_train, y_train)
print("Gaussian NB     churn F1:", round(f1_score(y_test, gnb.predict(X_test)), 3))
print("Logistic regr.  churn F1:", round(f1_score(y_test, pred_test), 3))
print()
print(classification_report(y_test, gnb.predict(X_test), target_names=["stay", "churn"]))

Gaussian NB     churn F1: 0.675
Logistic regr.  churn F1: 0.637

              precision    recall  f1-score   support

        stay       0.86      0.38      0.52       468
       churn       0.53      0.92      0.67       362

    accuracy                           0.61       830
   macro avg       0.70      0.65      0.60       830
weighted avg       0.72      0.61      0.59       830



Surprise: NB's churn F1 is *higher*. Read the whole report before celebrating. Its churn **recall** is 0.92 but its stay recall is only 0.38 and accuracy is 0.61: it is calling almost everyone a churner. A single-class F1 hides that; the macro average (0.60 against 0.67 for logistic regression) does not. Why so lopsided? `spend` and `orders` are heavily skewed, nothing like bell curves, so the fitted Gaussians for the *stay* class are wide and badly centred, and the features are far from independent (customers with many orders also have many products). Exercise 4 shows that a log transform fixes much of this.

## 7. Where Naive Bayes shines: text

Two thousand short product reviews, positive or negative. (Generated with a seed so this notebook needs no download; real review datasets arrive in Module 09.) `CountVectorizer` turns each review into word counts; `MultinomialNB` counts how often each word appears per class.

In [11]:
rng = np.random.default_rng(7)
pos_words = "great love perfect excellent happy recommend fast beautiful quality lovely".split()
neg_words = "broken refund disappointed terrible late damaged never waste poor awful".split()
neutral = "the order arrived box item product shop delivery colour size price".split()

def make_review(positive):
    sentiment = pos_words if positive else neg_words
    n_sent, n_neu = rng.integers(1, 4), rng.integers(3, 7)
    words = list(rng.choice(sentiment, n_sent)) + list(rng.choice(neutral, n_neu))
    if rng.random() < 0.15:                        # some reviews mix in an opposite word
        words.append(rng.choice(neg_words if positive else pos_words))
    rng.shuffle(words)
    return " ".join(words)

labels = rng.random(2000) < 0.6                    # 60% positive
reviews = pd.DataFrame({"text": [make_review(p) for p in labels], "positive": labels.astype(int)})
reviews.head()

,text,positive
0,product awful price item late,0
1,refund arrived box order terrible colour shop,0
2,item broken awful arrived shop shop,0
3,delivery product order excellent shop beautifu...,1
4,price arrived colour recommend arrived item,1


In [12]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split

Xr_train, Xr_test, yr_train, yr_test = train_test_split(reviews["text"], reviews["positive"], test_size=0.25, random_state=0)
vec = CountVectorizer()
counts_train = vec.fit_transform(Xr_train)          # rows = reviews, columns = words, values = counts
mnb = MultinomialNB(alpha=1).fit(counts_train, yr_train)
pred_r = mnb.predict(vec.transform(Xr_test))
print(classification_report(yr_test, pred_r, target_names=["negative", "positive"]))

              precision    recall  f1-score   support

    negative       0.99      0.98      0.99       192
    positive       0.99      1.00      0.99       308

    accuracy                           0.99       500
   macro avg       0.99      0.99      0.99       500
weighted avg       0.99      0.99      0.99       500



In [13]:
# Which words push hardest toward each class?  log P(word | positive) - log P(word | negative)
words = np.array(vec.get_feature_names_out())
log_ratio = mnb.feature_log_prob_[1] - mnb.feature_log_prob_[0]
order = np.argsort(log_ratio)
print("most negative:", ", ".join(words[order[:8]]))
print("most positive:", ", ".join(words[order[-8:]][::-1]))

most negative: disappointed, awful, never, waste, broken, terrible, damaged, poor
most positive: quality, happy, lovely, excellent, recommend, great, perfect, beautiful


In [14]:
test_email = ["the item arrived broken and late, I want a refund"]
print("P(negative, positive) =", mnb.predict_proba(vec.transform(test_email)).round(3))

P(negative, positive) = [[0.998 0.002]]


Every word in the email contributed one multiplied fraction, exactly as in the ten-row table, just with 30 features instead of two. That is why NB handles text so naturally: the maths does not care how many columns there are.

## Quiz answers

1. $P(\text{class})$: how common the class is before seeing any evidence (0.4 churn in the table).
2. It assumes the features are independent of each other given the class, which is almost never true.
3. A likelihood of exactly zero from an unseen feature value, which would make a class impossible no matter what the other features say.
4. **Multinomial NB** for counts (words); **Gaussian NB** for continuous measurements.
5. Any of: it only needs the ranking right, not the exact probability; it trains in one pass over the counts; it works with very few examples and tens of thousands of features.

**Next:** [03.6 The confusion matrix](../06-confusion-matrix-metrics/)